# TiN(111) / Si₃N₄(0001) Interface Builder (pymatgen)

Builds Ti-terminated **and** N-terminated `TiN(111) // Si₃N₄(0001)`
heterointerfaces for MLIP training data generation (VASP-MLFF, MACE, DeepMD),
at three atom-count scales -- a small, strict 250-400 atom cell for VASP-MLIP
training, a wider/thinner 250-400 atom variant with more interface area per
atom, and a large (thousands of atoms) cell for thermal-conductivity-scale
work -- each available either as a vacuum slab (a free film surface) or a
fully 3D-periodic sandwich (no vacuum, matched film/substrate contact on
*both* sides through the periodic boundary -- the wrap-around interface is
now explicitly rebuilt to carry the same termination as the primary one, not
just whatever the film's other face happens to be).

This uses `pymatgen.analysis.interfaces.CoherentInterfaceBuilder`, which is a
direct implementation of the Zur & McGill (1984) lattice-matching algorithm
plus slab generation and coherent-interface stacking -- the same tool used for
the original, smaller interface. This notebook is a thin layer on top of it:
build the two bulk structures, hand them to `CoherentInterfaceBuilder`, pick
out the Ti- and N-terminated results, tile to size, export.

**Pipeline**

1. Build bulk TiN (rock-salt) and β-Si₃N₄ (hexagonal) from crystallographic data.
2. Hand both to `CoherentInterfaceBuilder` with the `(111)`/`(0001)` Miller
   indices -- it finds the low-strain coincidence lattice, cuts the slabs, and
   builds the stacked interface.
3. Identify which of its terminations are Ti- vs. N-terminated on the film side.
4. Tile the matched cell up to any target atom count (thousands -> tens of
   thousands) -- exact, no additional strain, since the matched cell is already
   commensurate.
4b. Build a fully periodic, no-vacuum "sandwich" version of each interface too
   (a bolt-on on top of the same matched structures from steps 2-3), rebuilding
   the film at whatever thickness makes its top and bottom faces the same
   termination so the periodic wrap-around interface matches the primary one.
4c. Separately, build a small (250-400 atom) cell for VASP-MLIP training,
   with a much smaller ZSL search area and a loosened strain tolerance --
   a small coincidence cell generally can't also hit a tight strain fit.
4d. And a wider/thinner variant of that small cell (same atom budget, more
   interface area, fewer bulk-buffer layers) -- same machinery, different
   thickness/area parameters.
4e. Build partial-oxidation `TiN(1-x)O(x)` variants of the small MLIP cells
   (`x` = 0.25/0.5/0.75/1.0, both an idealized simple-substitution version and
   a "realistic" version with rock-salt TiO's intrinsic ~15% vacancy
   concentration included), plus standalone rock-salt TiO bulk references.
5. Export to `POSCAR` (VASP-MLFF) and `extxyz` (for your `gather_extxyz_mace.py`
   pipeline, via `pymatgen`'s ASE adaptor) -- the vacuum-slab, periodic,
   small-cell, and partial-oxidation versions.

> ⚠️ **Before production use:** the bulk lattice constants below are literature
> defaults. Since you've already generated relaxed bulk TiN and Si₃N₄ structures
> with VASP, swap `TIN_A`, `SI3N4_A`, `SI3N4_C` in the parameters cell for
> **your own relaxed values** so the interface is consistent with the rest of
> your training set.

> **Note:** this was written and syntax-checked in a sandbox without pymatgen/ASE
> installed, so it hasn't been execution-tested end-to-end on this end. If a
> keyword argument name doesn't match your installed pymatgen version (the
> `CoherentInterfaceBuilder`/`ZSLGenerator` signatures have shifted slightly
> across releases), `help(ZSLGenerator.__init__)` or
> `help(CoherentInterfaceBuilder.get_interfaces)` will show your version's
> exact signature -- send me the traceback and I'll fix it directly.


## Crystallographic data & sources

- **TiN**: rock-salt (Fm-3m), 2-atom basis. Experimental lattice constant
  `a ≈ 4.24 Å`. Replace with your VASP-relaxed value for MLIP-consistency.
- **β-Si₃N₄**: hexagonal, space group *P6₃/m* (No. 176), `a = 7.614 Å`,
  `c = 2.912 Å`, Wyckoff sites Si (6h), N1 (2c), N2 (6h)
  (Billy *et al.*, 1983; via the American Mineralogist Crystal Structure Database /
  RRUFF "nierite" entry: https://rruff.geo.arizona.edu/AMS/minerals/Nierite).
  This gives 6 Si + 8 N = 14 atoms per conventional cell (`Si6N8` = 2 × Si₃N₄), which
  is the correct stoichiometry -- sanity-checked below.
- Orientation relationship: `TiN(111) // Si3N4(0001)`, matching the interfaces you
  built previously.


In [ ]:
import itertools
import numpy as np
from pymatgen.core import Structure, Lattice
from pymatgen.analysis.interfaces.zsl import ZSLGenerator
from pymatgen.analysis.interfaces.coherent_interfaces import CoherentInterfaceBuilder
from pymatgen.io.ase import AseAtomsAdaptor
from ase.io import write
from scipy.spatial import cKDTree

np.set_printoptions(precision=4, suppress=True)

# ----------------------------- PARAMETERS ----------------------------- #
# --- Bulk lattice constants (REPLACE with your own VASP-relaxed values!) ---
TIN_A    = 4.24     # TiN rock-salt cubic lattice constant (A)
SI3N4_A  = 7.614    # beta-Si3N4 hexagonal a (A)
SI3N4_C  = 2.912    # beta-Si3N4 hexagonal c (A)

# --- Slab thickness (A), before vacuum ---
TIN_THICKNESS   = 25.0
SI3N4_THICKNESS = 25.0

# --- Interface / vacuum geometry ---
INTERFACE_GAP = 2.2   # starting Ti/N <-> Si/N contact distance (A), relax with DFT/MLIP after
VACUUM        = 15.0  # vacuum padding above the free surface (A)

# --- Zur-McGill (ZSL) lattice matching ---
ZSL_MAX_AREA        = 150.0  # A^2, largest matched-cell area to search -- raise this
                              # only if no match is found within ZSL_MAX_STRAIN
ZSL_MAX_STRAIN      = 0.02   # 2% max in-plane length/angle mismatch to accept a match

# --- Final size target ---
TARGET_ATOMS = 5000     # tile the matched interface up to (at least) this many atoms

# --- Small, strict-budget cell for VASP-MLIP training (separate from the big
# thermal-conductivity cell above) -- a small coincidence cell generally can't
# also hit a tight strain tolerance, so this loosens ZSL strain and searches a
# much smaller area, and uses thinner slabs to fit the atom budget.
SMALL_CELL_TARGET_ATOMS    = (250, 400)  # (min, max) atoms, inclusive
SMALL_CELL_TIN_THICKNESS   = 10.0   # A -- thinner than TIN_THICKNESS above
SMALL_CELL_SI3N4_THICKNESS = 10.0   # A -- thinner than SI3N4_THICKNESS above
SMALL_CELL_ZSL_START_AREA  = 40.0   # A^2 -- much smaller than ZSL_MAX_AREA, or
                                     # CoherentInterfaceBuilder will just find the
                                     # same big-cell-sized match again
SMALL_CELL_ZSL_MAX_STRAIN  = 0.08   # 8% -- loosened vs. ZSL_MAX_STRAIN=2%

# --- Same small-cell atom budget, different shape: wider in-plane area, thinner
# slabs (more interface area sampled per cell, fewer bulk-buffer layers away
# from the interface) ---
SMALL_CELL_WIDE_TIN_THICKNESS   = 6.0    # A -- thinner than SMALL_CELL_TIN_THICKNESS
SMALL_CELL_WIDE_SI3N4_THICKNESS = 6.0    # A -- thinner than SMALL_CELL_SI3N4_THICKNESS
SMALL_CELL_WIDE_ZSL_START_AREA  = 90.0   # A^2 -- larger than SMALL_CELL_ZSL_START_AREA
SMALL_CELL_WIDE_ZSL_MAX_STRAIN  = 0.05   # a larger area can usually afford a tighter fit

# --- Partial oxidation: TiN(1-x)O(x) via anion-site substitution on the
# rock-salt lattice. Ti(N,O) is a documented continuous rock-salt (B1)
# solid solution (O and N randomly disordered on the shared anion site),
# and rock-salt TiO is close to lattice-matched with TiN -- see the 4e
# markdown for sources and the approximations this makes. ---
TIO_A            = 4.18   # rock-salt (B1) TiO lattice constant (A), near-stoichiometric
TIO_VACANCY_FRAC = 0.15   # ~15% intrinsic structural vacancies on BOTH the Ti and O
                           # sublattices in near-stoichiometric rock-salt TiO (orders into
                           # a monoclinic Ti5O5 superstructure below ~950 C; this notebook
                           # uses the disordered/random-vacancy approximation). Well-
                           # established in the literature for TiO specifically.
TIN_VACANCY_FRAC = 0.05   # Baseline N-sublattice vacancy fraction assumed for "realistic"
                           # TiN (x=0), representing a WELL-OPTIMIZED, near-stoichiometric
                           # film/crystal -- not TiN's vacancy fraction in general, which has
                           # no single value (unlike TiO's ~15%). Real sputtered TiN spans
                           # N:Ti = 0.587-1.40 depending on deposition conditions (Chen et al.,
                           # https://doaj.org/article/30afc1597e7c4e5788321ea193fc85e5), with
                           # the N-vacancy phase experimentally stable up to ~63% N-deficiency
                           # and predicted metastable to ~80% (Holec et al. nanocon 2020,
                           # https://www.confer.cz/nanocon/2020/read/3706-...pdf) -- i.e. 0.05
                           # sits deliberately at the low, "good film" end of a MUCH wider
                           # real range, not a central/typical value. Ti sublattice is treated
                           # as fully occupied for pure TiN (non-stoichiometry in TiN is
                           # dominated by N vacancies). If you have a measured N:Ti ratio for
                           # your own material (RBS/XPS/EDS), use it directly instead:
                           # TIN_VACANCY_FRAC = 1 - (measured N:Ti ratio).
OXIDATION_X_POINTS = [0.25, 0.5, 0.75, 1.0]  # TiN(1-x)O(x) compositions to build
RANDOM_SEED = 42          # reproducible random substitution / vacancy placement

# --- Safety check ---
MIN_BOND_DIST = 1.0  # A -- any two atoms closer than this signal a real bug
                      # (duplicate/overlapping atoms), never a real Ti-N/Si-N bond


## 1. Bulk structures

In [ ]:
def build_tin_bulk(a=TIN_A):
    """Rock-salt TiN (Fm-3m). Structure.from_spacegroup expands the 2-site
    basis via the space group's symmetry operations into the conventional
    8-atom cell; CoherentInterfaceBuilder finds the minimal oriented cell for
    a given Miller index internally, so we don't need to hand it a primitive
    cell ourselves."""
    return Structure.from_spacegroup(
        "Fm-3m", Lattice.cubic(a), ["Ti", "N"], [[0, 0, 0], [0.5, 0, 0]]
    )


def build_si3n4_bulk(a=SI3N4_A, c=SI3N4_C):
    """beta-Si3N4, hexagonal P6_3/m (#176), Billy et al. 1983 Wyckoff positions."""
    return Structure.from_spacegroup(
        176,
        Lattice.hexagonal(a, c),
        ["Si", "N", "N"],
        [
            [0.1799, 0.7733, 0.25],  # Si  -- Wyckoff 6h
            [1 / 3, 2 / 3, 0.25],    # N1  -- Wyckoff 2c
            [0.3356, 0.0330, 0.25],  # N2  -- Wyckoff 6h
        ],
    )


tin_bulk = build_tin_bulk()
si3n4_bulk = build_si3n4_bulk()

for name, st in [("TiN", tin_bulk), ("Si3N4", si3n4_bulk)]:
    print(f"{name:8s} formula={st.composition.reduced_formula:10s} natoms={len(st):3d} "
          f"cell={st.lattice.parameters}  density={st.density:.3f} g/cm^3")

n_si = sum(1 for sp in si3n4_bulk.species if sp.symbol == "Si")
n_n = sum(1 for sp in si3n4_bulk.species if sp.symbol == "N")
assert n_n / n_si == 4 / 3, f"Si3N4 stoichiometry check failed: Si={n_si} N={n_n}"
print("Si3N4 stoichiometry OK: Si:N =", n_si, ":", n_n)


## 2. Coherent interface matching + construction

`CoherentInterfaceBuilder` takes both *bulk* structures and the Miller indices
directly -- it does the slab cutting, Zur-McGill lattice matching, straining,
and stacking internally. `.terminations` lists every distinct (film, substrate)
termination pair it found; we build one representative interface per pair below
and classify each by inspecting which element actually sits at the film side of
the interface (rather than trusting the termination label strings, whose exact
format varies by pymatgen version).

Two things tuned here based on a real run:

- `filter_out_sym_slabs=False` -- pymatgen's default (`True`) discards any
  surface termination that isn't symmetric on its own, which for a rock-salt
  (111) cut tends to keep only *one* of the two physical terminations (Ti- or
  N-terminated) and silently drop the other -- exactly the failure seen (only
  1 termination pair found, when TiN(111) should offer both).
- `ZSL_MAX_AREA` needs headroom above the actual matched-cell area, not just
  to equal it: the coincidence cell found by hand earlier works out to
  ~149 A^2, and `ZSL_MAX_AREA=150` was cutting that too close for pymatgen's
  own (not necessarily identical) area accounting, producing "No ZSL matches
  found." `build_cib()` below retries with a growing area cap instead of
  guessing a single fixed number.


In [ ]:
def build_cib(film, substrate, film_miller, substrate_miller,
              start_area=ZSL_MAX_AREA, max_strain=ZSL_MAX_STRAIN,
              area_growth=2.0, max_tries=6):
    """Build a CoherentInterfaceBuilder, growing the ZSL search area until at
    least one match is found (rather than guessing a single fixed area cap)."""
    area = start_area
    for attempt in range(1, max_tries + 1):
        zsl = ZSLGenerator(max_area=area, max_length_tol=max_strain, max_angle_tol=max_strain)
        cib = CoherentInterfaceBuilder(
            film_structure=film,
            substrate_structure=substrate,
            film_miller=film_miller,
            substrate_miller=substrate_miller,
            zslgen=zsl,
            filter_out_sym_slabs=False,
        )
        n_matches = len(cib.zsl_matches)
        print(f"  attempt {attempt}: max_area={area:.1f} A^2 -> {n_matches} ZSL match(es), "
              f"{len(cib.terminations)} termination pair(s)")
        if n_matches > 0:
            return cib
        area *= area_growth
    raise ValueError(
        f"No ZSL matches found even at max_area={area:.1f} A^2 -- try also "
        "loosening ZSL_MAX_STRAIN.")


cib = build_cib(tin_bulk, si3n4_bulk, (1, 1, 1), (0, 0, 1))

print(f"\n{len(cib.terminations)} termination pair(s) found:")
for t in cib.terminations:
    print(" ", t)


## 3. Identify Ti- and N-terminated interfaces

For each termination pair, build one interface (pymatgen's ZSL search already
orders candidates best-match-first, so we take the first one it yields) at the
target thickness/gap/vacuum, then classify it: cluster atoms into z-layers,
find the largest z-gap (that's the film/substrate interface, since it's the
only gap of order `INTERFACE_GAP`+vacuum-scale, much bigger than any real
interlayer spacing), and check the composition of the film-side layer
immediately above it.


In [ ]:
def get_z_layers(z, tol=0.3):
    """Cluster (sorted) z-values into layers. Returns a list of index-lists
    into the original (unsorted) z array."""
    order = np.argsort(z)
    layers = [[order[0]]]
    for idx in order[1:]:
        if z[idx] - z[layers[-1][-1]] > tol:
            layers.append([idx])
        else:
            layers[-1].append(idx)
    return layers


def assert_no_overlaps(cart_coords, min_dist=MIN_BOND_DIST, label=""):
    """Hard safety check: no two atoms should ever be closer than min_dist --
    a real Ti-N/Si-N bond is ~1.7-2.1 A, so anything under 1.0 A means
    duplicate/overlapping atoms, not physics."""
    tree = cKDTree(cart_coords)
    close = tree.query_pairs(r=min_dist)
    if close:
        i, j = next(iter(close))
        d = np.linalg.norm(cart_coords[i] - cart_coords[j])
        raise ValueError(
            f"{label}: {len(close)} atom pair(s) closer than {min_dist} A "
            f"(e.g. atoms {i},{j} at {d:.3f} A) -- unphysical overlap/duplicate, "
            "do not use this structure.")
    print(f"  OK [{label}]: no atoms closer than {min_dist} A ({len(cart_coords)} atoms)")


def classify_film_termination(interface, tol=0.3):
    """Return the element symbol at the film side of the interface (the
    layer immediately above the single largest z-gap), or None if that layer
    isn't a clean single element."""
    z = interface.cart_coords[:, 2]
    species = np.array([sp.symbol for sp in interface.species])
    layers = get_z_layers(z, tol=tol)
    layer_z = [z[l].mean() for l in layers]
    gaps = np.diff(layer_z)
    split = int(np.argmax(gaps))  # index of the layer just below the biggest gap
    film_layer = layers[split + 1]
    elems = set(species[film_layer])
    return elems.pop() if len(elems) == 1 else None


interfaces_raw = {}
for term in cib.terminations:
    gen = cib.get_interfaces(
        termination=term,
        gap=INTERFACE_GAP,
        vacuum_over_film=VACUUM,
        film_thickness=TIN_THICKNESS,
        substrate_thickness=SI3N4_THICKNESS,
        in_layers=False,
    )
    iface = next(gen, None)
    if iface is None:
        print(f"  {term}: no interface generated, skipping")
        continue
    label = classify_film_termination(iface)
    print(f"  {term} -> film-side element at interface: {label}  ({len(iface)} atoms)")
    interfaces_raw[term] = (iface, label)

interfaces = {}
interface_terms = {}  # name -> the cib.terminations entry that produced it
                       # (kept so 4b can rebuild at a different film_thickness)
for term, (iface, label) in interfaces_raw.items():
    if label == "Ti" and "Ti-term" not in interfaces:
        interfaces["Ti-term"] = iface
        interface_terms["Ti-term"] = term
    elif label == "N" and "N-term" not in interfaces:
        interfaces["N-term"] = iface
        interface_terms["N-term"] = term

for name, iface in interfaces.items():
    assert_no_overlaps(iface.cart_coords, label=f"{name} interface (unit cell)")
    print(f"{name}: {len(iface)} atoms, formula={iface.composition.reduced_formula}, "
          f"cell a,b={iface.lattice.a:.3f},{iface.lattice.b:.3f} A")

missing = {"Ti-term", "N-term"} - interfaces.keys()
if missing:
    print(f"\\nWARNING: could not find a clean single-element interface for: {missing}. "
          "Inspect `interfaces_raw` above -- CoherentInterfaceBuilder may not have "
          "produced a single-element-terminated match within ZSL_MAX_AREA/ZSL_MAX_STRAIN; "
          "try loosening those, or inspect classify_film_termination's output by hand.")


## 4. Scale up to thousands of atoms

The matched coincidence cell already carries the residual strain; tiling it by
integer repeats in-plane is exact and adds **no additional strain**, so this is
the cheap way to reach large atom counts for MD/MLIP training cells.
`Structure.__mul__` (pymatgen's own, well-tested supercell operator) does the
tiling.


In [ ]:
def scale_to_atoms(structure, target_atoms=TARGET_ATOMS, max_side=15, max_overshoot=1.6):
    """Tile (nx, ny) to reach >= target_atoms, preferring a square-ish cell
    over the fewest possible atoms."""
    n_unit = len(structure)
    candidates = []
    for nx in range(1, max_side + 1):
        for ny in range(1, max_side + 1):
            n = n_unit * nx * ny
            if n < target_atoms:
                continue
            aspect = max(nx, ny) / min(nx, ny)
            candidates.append((n / target_atoms, aspect, n, nx, ny))
    if not candidates:
        raise ValueError("Could not reach target atom count within max_side; "
                          "increase max_side or lower target_atoms.")
    within = [c for c in candidates if c[0] <= max_overshoot]
    pool = within if within else candidates
    pool.sort(key=lambda c: (c[1], c[0]))  # most square first, then least overshoot
    _, aspect, n, nx, ny = pool[0]
    big = structure * (nx, ny, 1)
    print(f"  tiling {nx} x {ny} (aspect {aspect:.2f}): {n_unit} -> {len(big)} atoms "
          f"(cell a,b = {big.lattice.a:.2f}, {big.lattice.b:.2f} A)")
    return big


large_interfaces = {}
for name, iface in interfaces.items():
    print(name)
    big = scale_to_atoms(iface, TARGET_ATOMS)
    assert_no_overlaps(big.cart_coords, label=f"{name} interface (tiled)")
    large_interfaces[name] = big


## 4b. Fully periodic ("sandwich") interfaces -- no vacuum

Everything above is a vacuum slab: substrate, the controlled interface, film,
then vacuum above the free film surface -- not periodic in z (the vacuum gap
is empty space, not a second interface). This section builds an *additional*,
fully 3D-periodic version of each interface for cases that need real
z-periodicity (e.g. stress-tensor-based MLIP training, or avoiding a fake free
surface entirely): the vacuum is replaced with a second film/substrate contact
at the same `INTERFACE_GAP`, formed by the periodic boundary itself -- so
going around the cell in z you get `...substrate-film-substrate-film...`,
matched at the same gap distance on both sides, not `...substrate-film-vacuum...`.

This is a bolt-on, not a re-derivation: it starts from the exact structures
`CoherentInterfaceBuilder` already matched, strained, and stacked above (the
hard part), and only trims the vacuum and resets the cell's `c` length so the
film's free (top) face reconnects to the substrate's free (bottom) face
through the periodic boundary. Nothing above (the vacuum-slab `interfaces` /
`large_interfaces`) is changed or replaced by this.

**Termination-matching fix (was a caveat, now handled):** the *original*
interface (substrate-top meets film-bottom) has its in-plane registry
optimized by `CoherentInterfaceBuilder`'s shift search. The periodic-boundary
interface it creates once vacuum is stripped (film-top meets substrate-bottom)
does *not* get that treatment for free -- it's whatever results from the
film's *other*, uncontrolled face. Confirmed on a real run: TiN(111)
alternates single-element Ti/N atomic planes, so with the plain thickness-in-A
slicing an *even* number of layers puts **opposite** elements on the film's
two faces -- a "Ti-term" cell came back Ti-terminated at the built interface
but N-terminated at the periodic wrap-around seam, i.e. the two interfaces in
the same cell were not actually equivalent.

`match_film_faces()` below fixes this: it trims exactly one atomic layer off
the **top** (never the bottom) of the interface you already built and
already validated as the vacuum-slab version, if and only if the top face
doesn't already match the bottom -- since TiN(111) alternates single-element
planes, removing one plane always flips an even total count (mismatched) to
odd (matched). *Both* interfaces then end up the same termination (e.g. both
Ti-Si₃N₄ contacts), and -- importantly -- the bottom face, i.e. the primary,
`termination`-controlled interface, is never touched, so it can't drift away
from what you asked for.

(An earlier version of this fix instead rebuilt the interface via
`cib.get_interfaces()` at a larger `film_thickness`. A real run showed that
approach is unsafe: changing the Angstrom thickness, for the *same*
`termination` label, changed *which* element ended up at the bottom face --
not just the top -- so a "Ti-term" cell came back with an *N-terminated*
primary interface. Trimming an already-built interface avoids that failure
mode entirely, since nothing is ever rebuilt through `cib`.)

Each periodic section below (4b, 4c, 4d) builds **both** versions and exports
**both**, so you can directly compare the actual POSCAR/extxyz files, not
just printed output: `periodic_interfaces` / `large_periodic_interfaces`
(etc.) are the fixed, **equivalent**-termination version used everywhere else
in this notebook; `periodic_interfaces_inequivalent` /
`large_periodic_interfaces_inequivalent` (etc.) are the *pre-fix* build --
plain `make_fully_periodic()` on the unmodified vacuum interface, no
thickness search -- exported with an `_inequivalent` suffix purely for
side-by-side comparison against the fixed version. Don't use the
`_inequivalent` files for actual training data; they exist only so you can
see what the fix changed.

**Is the wrap-around ("top") interface actually correct?** A plain,
non-periodic overlap check (`assert_no_overlaps`) genuinely cannot see this
seam -- the top and bottom layers are ~the slab's full thickness apart in the
raw, unwrapped coordinates, even though they're `gap`-close once you account
for periodicity. `check_periodic_boundary()` below does the check that
actually matters here: it explicitly shifts the top layer's periodic image
down by one c-vector and measures its real closest-contact distance to the
bottom layer, then either raises (if that distance is under `MIN_BOND_DIST`
-- a real overlap) or flags it (if it's much larger than the intended `gap`
-- a poor-registry seam, not an error, but worth a visual check before using
this structure for training data).


In [ ]:
def make_fully_periodic(interface, gap=INTERFACE_GAP):
    """Strip the vacuum from a CoherentInterfaceBuilder interface and set the
    c-vector so the structure is fully 3D-periodic: the film's free (top) face
    reconnects to the substrate's free (bottom) face through the periodic
    boundary with `gap` spacing -- the same contact distance as the original,
    controlled interface. Doesn't touch a,b (in-plane) or any atom's x,y,z
    within the slab -- only crops vacuum out of the cell's c length."""
    coords = interface.cart_coords.copy()
    z = coords[:, 2]
    z0, z1 = z.min(), z.max()
    thickness = z1 - z0
    coords[:, 2] -= z0  # atoms now span exactly [0, thickness]

    lattice = interface.lattice.matrix.copy()
    c_norm = np.linalg.norm(lattice[2])
    c_dir = lattice[2] / c_norm
    assert np.allclose(np.abs(c_dir), [0, 0, 1], atol=1e-6), (
        "expected c-vector along z -- this interface's cell isn't in the "
        "orthogonal-c convention this function assumes.")
    lattice[2] = c_dir * (thickness + gap)

    return Structure(lattice, interface.species, coords, coords_are_cartesian=True)


def check_periodic_boundary(periodic, gap=INTERFACE_GAP, tol=0.3,
                             min_dist=MIN_BOND_DIST, far_factor=3.0, label=""):
    """Report AND verify the wrap-around interface (bottom layer <-> top
    layer, connected through the periodic c-axis boundary): shift the top
    layer's periodic image down by one c-vector and measure its true closest-
    contact distance to the bottom layer -- this is the number that actually
    answers 'is the top interface correct', not just which species are
    present. Raises on a real overlap (< min_dist); flags (doesn't raise) a
    seam that's suspiciously far from the intended gap, since a bad registry
    is a quality issue to inspect, not a hard bug."""
    coords = periodic.cart_coords
    z = coords[:, 2]
    c_len = periodic.lattice.matrix[2, 2]
    species = np.array([sp.symbol for sp in periodic.species])
    layers = get_z_layers(z, tol=tol)
    bottom_idx, top_idx = layers[0], layers[-1]

    top_shifted = coords[top_idx].copy()
    top_shifted[:, 2] -= c_len  # bring the top layer's periodic image just below z=0
    tree = cKDTree(coords[bottom_idx])
    dists, _ = tree.query(top_shifted, k=1)
    min_d = dists.min()

    print(f"  [{label}] periodic-boundary interface: bottom {sorted(set(species[bottom_idx]))} "
          f"<-> top {sorted(set(species[top_idx]))}, closest contact = {min_d:.3f} A "
          f"(target gap = {gap} A)")
    if min_d < min_dist:
        raise ValueError(
            f"[{label}] periodic-boundary interface has atoms {min_d:.3f} A apart -- "
            f"unphysical overlap (< {min_dist} A), do not use this structure.")
    if min_d > gap * far_factor:
        print(f"    NOTE: closest contact ({min_d:.3f} A) is much larger than the "
              f"target gap ({gap} A) -- these two faces likely aren't in good registry "
              "at this seam (expected, per the caveat above); inspect visually before "
              "using this for training data.")
    return min_d


def get_top_layer_elements(interface, tol=0.3):
    """Element symbols present in the highest-z atomic layer of `interface`."""
    z = interface.cart_coords[:, 2]
    species = np.array([sp.symbol for sp in interface.species])
    top_layer = get_z_layers(z, tol=tol)[-1]
    return set(species[top_layer])


def trim_top_layer(interface, tol=0.3):
    """Remove the single highest-z atomic layer from `interface`. Used to
    flip a film's top/bottom-face parity by exactly one atomic plane."""
    z = interface.cart_coords[:, 2]
    top_idx = set(get_z_layers(z, tol=tol)[-1])
    keep = [i for i in range(len(interface)) if i not in top_idx]
    coords = interface.cart_coords[keep]
    all_species = interface.species  # cache: pymatgen rebuilds this list on
                                      # every access, so index into a cached
                                      # copy instead of re-fetching per-atom
    species = [all_species[i] for i in keep]
    return Structure(interface.lattice.matrix.copy(), species, coords,
                      coords_are_cartesian=True)


def match_film_faces(interface, target_element, tol=0.3, max_trims=6):
    """Trim atomic layers off the TOP of `interface` (one at a time) until
    the film's top face is the same element as its bottom (interface-facing)
    face -- confirmed on a real run: TiN(111) alternates single-element
    Ti/N atomic planes, so trimming exactly one plane always flips an even
    total plane count (top != bottom) to odd (top == bottom).

    An earlier version of this fix instead rebuilt the interface via
    `cib.get_interfaces()` at a larger `film_thickness`. A real run showed
    that doesn't work: changing the Angstrom thickness, for the *same*
    `termination` label, changed *which* element ended up at the bottom
    (interface-facing) face -- not just the top -- so the "fixed" cell came
    back with the wrong primary interface entirely (N-terminated instead of
    the requested Ti-terminated). This version never rebuilds anything via
    `cib` -- it only removes atoms from the top of the interface you already
    built (and already validated/exported as the vacuum-slab version), so
    the primary, controlled interface can never change; only the free top
    face is touched."""
    current = interface
    for _ in range(max_trims):
        bottom = classify_film_termination(current, tol=tol)
        top = get_top_layer_elements(current, tol=tol)
        if bottom == target_element and top == {target_element}:
            return current
        if bottom != target_element:
            print(f"    WARNING: film bottom face is {bottom}, expected "
                  f"{target_element} -- match_film_faces only trims the top "
                  "and can't fix a wrong bottom; returning as-is, inspect "
                  "the input interface.")
            return current
        current = trim_top_layer(current, tol=tol)
    print(f"    WARNING: top/bottom termination still doesn't match after "
          f"{max_trims} trims (target={target_element}) -- the periodic "
          "wrap-around seam for this cell may still be a different "
          "termination than the built one.")
    return current


def build_matched_periodic_small_cell(interface, target_element,
                                       target_range=SMALL_CELL_TARGET_ATOMS,
                                       tol=0.3):
    """match_film_faces(), then re-tile back into target_range if trimming
    (or the original build) left the atom count under budget -- mirrors
    build_small_cell's tiling logic. Reused as-is for both the 4c (narrow)
    and 4d (wide) small periodic cells."""
    matched = match_film_faces(interface, target_element, tol=tol)
    lo, hi = target_range
    n = len(matched)
    if n > hi:
        print(f"    WARNING: parity-matched small periodic cell is already "
              f"{n} atoms (> {hi}) -- using it as-is.")
        return matched
    if n < lo:
        print(f"    parity-matched small periodic cell is {n} atoms (< {lo}); "
              "tiling up:")
        matched = scale_to_atoms(matched, target_atoms=lo, max_side=3, max_overshoot=hi / lo)
    return matched


periodic_interfaces = {}
for name, iface in interfaces.items():
    target_element = name.split("-")[0]  # "Ti-term" -> "Ti", "N-term" -> "N"
    matched = match_film_faces(iface, target_element)
    assert_no_overlaps(matched.cart_coords,
                        label=f"{name} periodic interface (parity-matched, pre-wrap)")
    periodic = make_fully_periodic(matched)
    assert_no_overlaps(periodic.cart_coords, label=f"{name} periodic interface (unit cell)")
    check_periodic_boundary(periodic, label=name)
    periodic_interfaces[name] = periodic

large_periodic_interfaces = {}
for name, periodic in periodic_interfaces.items():
    print(name)
    big = scale_to_atoms(periodic, TARGET_ATOMS)
    assert_no_overlaps(big.cart_coords, label=f"{name} periodic interface (tiled)")
    large_periodic_interfaces[name] = big


# --- EQUIVALENT vs. INEQUIVALENT comparison build ---
# `periodic_interfaces` / `large_periodic_interfaces` above (the ones used
# everywhere else in this notebook) are the FIXED, parity-matched version --
# both the built interface and the periodic wrap-around seam carry the same
# termination ("equivalent"). This block rebuilds the *pre-fix* version --
# `make_fully_periodic()` applied directly to the plain vacuum interface at
# TIN_THICKNESS, with no thickness search -- purely so you can diff the two
# `check_periodic_boundary()` printouts and see exactly what the fix changed.
# Not used by any other section; kept only for this comparison.
periodic_interfaces_inequivalent = {}
for name, iface in interfaces.items():
    periodic = make_fully_periodic(iface)
    assert_no_overlaps(periodic.cart_coords,
                        label=f"{name} periodic interface, INEQUIVALENT (unit cell)")
    check_periodic_boundary(periodic, label=f"{name} (inequivalent)")
    periodic_interfaces_inequivalent[name] = periodic

large_periodic_interfaces_inequivalent = {}
for name, periodic in periodic_interfaces_inequivalent.items():
    print(name)
    big = scale_to_atoms(periodic, TARGET_ATOMS)
    assert_no_overlaps(big.cart_coords,
                        label=f"{name} periodic interface INEQUIVALENT (tiled)")
    large_periodic_interfaces_inequivalent[name] = big


## 4c. Small, strict-budget cell for VASP-MLIP training

The big cell above (thousands of atoms) is for thermal-conductivity-scale MD;
VASP-MLFF training needs something much smaller -- 250-400 atoms at most.
That's a hard constraint the ZSL match has to fit *under*, not a target to
tile up to, so this reuses `build_cib()` from section 2 with a much smaller
search area (`SMALL_CELL_ZSL_START_AREA`) and thinner slabs
(`SMALL_CELL_TIN_THICKNESS` / `SMALL_CELL_SI3N4_THICKNESS`) so the smallest
available coincidence cell is actually small enough to matter.

The tradeoff: a small coincidence cell generally *can't* also hit a tight
strain tolerance (fewer candidate (p,q) lattice-vector combinations exist
within a small area, so the best available match is usually worse) --
`SMALL_CELL_ZSL_MAX_STRAIN` is loosened to 8% (vs. 2% for the big cell)
accordingly. `cib.zsl_matches` comes back area-ascending, so the *first*
match for a given termination is the smallest one available; if that's still
under the 250-atom floor, it's tiled up with the exact same
`scale_to_atoms()` used for the big cell (just anchored at 250 instead of
5000, with the overshoot cap set so it can't tile past 400). If even the
smallest available match is already over 400 atoms, that's reported instead
of silently handing back an oversized "small" cell -- shrink
`SMALL_CELL_ZSL_START_AREA` or the small-cell thicknesses further in that case.


In [ ]:
small_cib = build_cib(
    tin_bulk, si3n4_bulk, (1, 1, 1), (0, 0, 1),
    start_area=SMALL_CELL_ZSL_START_AREA, max_strain=SMALL_CELL_ZSL_MAX_STRAIN,
)

print(f"\n{len(small_cib.terminations)} termination pair(s) found for the small cell:")
for t in small_cib.terminations:
    print(" ", t)


def build_small_cell(cib, termination, target_range=SMALL_CELL_TARGET_ATOMS,
                      gap=INTERFACE_GAP, vacuum=VACUUM,
                      film_thickness=SMALL_CELL_TIN_THICKNESS,
                      substrate_thickness=SMALL_CELL_SI3N4_THICKNESS):
    """Take the smallest-area ZSL match for `termination` and, if it's under
    target_range's minimum, tile it up with scale_to_atoms (capped so it can't
    overshoot the maximum). If even the smallest match already exceeds the
    maximum, return it anyway but flag it clearly -- shrinking further is a
    parameter change (area/thickness), not something to silently paper over."""
    lo, hi = target_range
    gen = cib.get_interfaces(
        termination=termination, gap=gap, vacuum_over_film=vacuum,
        film_thickness=film_thickness, substrate_thickness=substrate_thickness,
        in_layers=False,
    )
    iface = next(gen, None)
    if iface is None:
        return None
    n = len(iface)
    if n > hi:
        print(f"    WARNING: smallest available match is already {n} atoms "
              f"(> {hi}) -- reduce SMALL_CELL_ZSL_START_AREA or the small-cell "
              "thicknesses; using it as-is for now.")
        return iface
    if n < lo:
        print(f"    smallest match is {n} atoms (< {lo}); tiling up:")
        iface = scale_to_atoms(iface, target_atoms=lo, max_side=3, max_overshoot=hi / lo)
    return iface


small_interfaces_raw = {}
for term in small_cib.terminations:
    print(term)
    iface = build_small_cell(small_cib, term)
    if iface is None:
        print(f"  {term}: no interface generated, skipping")
        continue
    label = classify_film_termination(iface)
    print(f"  -> film-side element at interface: {label}  ({len(iface)} atoms)")
    small_interfaces_raw[term] = (iface, label)

small_interfaces = {}
small_interface_terms = {}
for term, (iface, label) in small_interfaces_raw.items():
    if label == "Ti" and "Ti-term" not in small_interfaces:
        small_interfaces["Ti-term"] = iface
        small_interface_terms["Ti-term"] = term
    elif label == "N" and "N-term" not in small_interfaces:
        small_interfaces["N-term"] = iface
        small_interface_terms["N-term"] = term

for name, iface in small_interfaces.items():
    assert_no_overlaps(iface.cart_coords, label=f"{name} small MLIP-training cell")
    n = len(iface)
    flag = "" if SMALL_CELL_TARGET_ATOMS[0] <= n <= SMALL_CELL_TARGET_ATOMS[1] else "  <-- OUT OF RANGE"
    print(f"{name}: {n} atoms{flag}, formula={iface.composition.reduced_formula}, "
          f"cell a,b={iface.lattice.a:.3f},{iface.lattice.b:.3f} A")

missing = {"Ti-term", "N-term"} - small_interfaces.keys()
if missing:
    print(f"\nWARNING: no small-cell match for {missing} within "
          f"{SMALL_CELL_TARGET_ATOMS} atoms -- try loosening SMALL_CELL_ZSL_MAX_STRAIN "
          "further, or reducing SMALL_CELL_TIN_THICKNESS/SMALL_CELL_SI3N4_THICKNESS.")


The small cell needs to be fully periodic too (same reasoning as section 4b),
including the same termination-matching fix: it uses
`build_matched_periodic_small_cell()` (from 4b) rather than reusing
`small_interfaces` directly, since the plain thickness-in-A build generally
won't already have matching top/bottom film faces. That rebuild can change
the atom count slightly (a whole extra atomic layer may get added to fix the
parity), so unlike a naive vacuum-strip this **can** need re-tiling to land
back in `SMALL_CELL_TARGET_ATOMS` -- `build_matched_periodic_small_cell`
handles that the same way `build_small_cell` does.

Also builds `small_periodic_interfaces_inequivalent` -- the pre-fix version,
for direct comparison, same as section 4b.


In [ ]:
small_periodic_interfaces = {}
for name, iface in small_interfaces.items():
    target_element = name.split("-")[0]
    matched = build_matched_periodic_small_cell(iface, target_element)
    assert_no_overlaps(matched.cart_coords,
                        label=f"{name} small periodic cell (parity-matched, pre-wrap)")
    periodic = make_fully_periodic(matched)
    assert_no_overlaps(periodic.cart_coords, label=f"{name} small periodic cell")
    check_periodic_boundary(periodic, label=f"{name} (small)")
    small_periodic_interfaces[name] = periodic
    print(f"{name}: {len(periodic)} atoms (periodic, no vacuum)")


# --- comparison build (see section 4b) -- pre-fix, plain make_fully_periodic
# on the unmodified small_interfaces, no thickness search ---
small_periodic_interfaces_inequivalent = {}
for name, iface in small_interfaces.items():
    periodic = make_fully_periodic(iface)
    assert_no_overlaps(periodic.cart_coords,
                        label=f"{name} small periodic cell, INEQUIVALENT")
    check_periodic_boundary(periodic, label=f"{name} (small, inequivalent)")
    small_periodic_interfaces_inequivalent[name] = periodic
    print(f"{name}: {len(periodic)} atoms (periodic, no vacuum, INEQUIVALENT)")


## 4d. Wider, thinner small cell (more interface area, fewer bulk layers)

Same 250-400 atom budget as 4c, different shape: `SMALL_CELL_WIDE_*` trades
slab thickness for in-plane area (`SMALL_CELL_WIDE_TIN_THICKNESS` /
`SMALL_CELL_WIDE_SI3N4_THICKNESS` = 6 A vs. 10 A in 4c, and
`SMALL_CELL_WIDE_ZSL_START_AREA` = 90 A^2 vs. 40 A^2), so within the same
atom count more of the cell is interface area and fewer atoms are spent on
bulk-buffer layers far from the interface. This is a pure bolt-on: it reuses
`build_cib`, `build_small_cell`, `classify_film_termination`,
`build_matched_periodic_small_cell`, `make_fully_periodic`, and
`check_periodic_boundary` unchanged -- only the parameters differ from 4c.


In [ ]:
wide_cib = build_cib(
    tin_bulk, si3n4_bulk, (1, 1, 1), (0, 0, 1),
    start_area=SMALL_CELL_WIDE_ZSL_START_AREA, max_strain=SMALL_CELL_WIDE_ZSL_MAX_STRAIN,
)

print(f"\n{len(wide_cib.terminations)} termination pair(s) found for the wide small cell:")
for t in wide_cib.terminations:
    print(" ", t)

wide_interfaces_raw = {}
for term in wide_cib.terminations:
    print(term)
    iface = build_small_cell(
        wide_cib, term,
        film_thickness=SMALL_CELL_WIDE_TIN_THICKNESS,
        substrate_thickness=SMALL_CELL_WIDE_SI3N4_THICKNESS,
    )
    if iface is None:
        print(f"  {term}: no interface generated, skipping")
        continue
    label = classify_film_termination(iface)
    print(f"  -> film-side element at interface: {label}  ({len(iface)} atoms)")
    wide_interfaces_raw[term] = (iface, label)

wide_interfaces = {}
wide_interface_terms = {}
for term, (iface, label) in wide_interfaces_raw.items():
    if label == "Ti" and "Ti-term" not in wide_interfaces:
        wide_interfaces["Ti-term"] = iface
        wide_interface_terms["Ti-term"] = term
    elif label == "N" and "N-term" not in wide_interfaces:
        wide_interfaces["N-term"] = iface
        wide_interface_terms["N-term"] = term

for name, iface in wide_interfaces.items():
    assert_no_overlaps(iface.cart_coords, label=f"{name} wide small MLIP-training cell")
    n = len(iface)
    flag = "" if SMALL_CELL_TARGET_ATOMS[0] <= n <= SMALL_CELL_TARGET_ATOMS[1] else "  <-- OUT OF RANGE"
    print(f"{name}: {n} atoms{flag}, formula={iface.composition.reduced_formula}, "
          f"cell a,b={iface.lattice.a:.3f},{iface.lattice.b:.3f} A")

missing = {"Ti-term", "N-term"} - wide_interfaces.keys()
if missing:
    print(f"\nWARNING: no wide small-cell match for {missing} within "
          f"{SMALL_CELL_TARGET_ATOMS} atoms -- try loosening SMALL_CELL_WIDE_ZSL_MAX_STRAIN "
          "further, or reducing SMALL_CELL_WIDE_TIN_THICKNESS/SMALL_CELL_WIDE_SI3N4_THICKNESS.")

wide_periodic_interfaces = {}
for name, iface in wide_interfaces.items():
    target_element = name.split("-")[0]
    matched = build_matched_periodic_small_cell(iface, target_element)
    assert_no_overlaps(matched.cart_coords,
                        label=f"{name} wide periodic cell (parity-matched, pre-wrap)")
    periodic = make_fully_periodic(matched)
    assert_no_overlaps(periodic.cart_coords, label=f"{name} wide periodic cell")
    check_periodic_boundary(periodic, label=f"{name} (wide)")
    wide_periodic_interfaces[name] = periodic
    print(f"{name}: {len(periodic)} atoms (periodic, no vacuum)")


# --- comparison build (see section 4b) -- pre-fix, plain make_fully_periodic
# on the unmodified wide_interfaces, no thickness search ---
wide_periodic_interfaces_inequivalent = {}
for name, iface in wide_interfaces.items():
    periodic = make_fully_periodic(iface)
    assert_no_overlaps(periodic.cart_coords,
                        label=f"{name} wide periodic cell, INEQUIVALENT")
    check_periodic_boundary(periodic, label=f"{name} (wide, inequivalent)")
    wide_periodic_interfaces_inequivalent[name] = periodic
    print(f"{name}: {len(periodic)} atoms (periodic, no vacuum, INEQUIVALENT)")


## 4e. Partial oxidation: TiN(1-x)O(x) film variants

**Is anion-site substitution valid here, or is TiO2 too different a phase?**
Rutile/anatase TiO2 -- yes, too different (tetragonal, not rock-salt; forming
that would mean a genuinely separate oxide *scale*, modeled as a second
interface, not a substitution). But **rock-salt TiO** (the *monoxide*, B1,
Fm-3m, same structure as TiN, `a ≈ 4.18 A` vs. TiN's `4.24 A`) is a different
story: titanium oxynitride Ti(N,O) is documented experimentally as a
**continuous rock-salt solid solution** across the full N/O ratio, with O and
N randomly disordered on the shared anion sublattice (Zhu *et al.*,
*J. Mater. Chem. A* 2016, "Titanium oxynitride microspheres with the
rock-salt structure..." -- https://pubs.rsc.org/en/content/articlehtml/2016/ta/c5ta06758h).
So `TiN(1-x)O(x)` built by substituting N -> O on the existing rock-salt film
lattice, for `x` in `OXIDATION_X_POINTS = [0.25, 0.5, 0.75, 1.0]`, is a valid
and standard approximation of partial TiN oxidation -- *if* what's being
modeled is oxygen incorporation into the TiN lattice (an oxynitride solid
solution), not a distinct TiO2 surface scale.

**Two versions, per your request:**

- **Idealized:** plain random N -> O substitution on the film's anion sites
  (species swap only, no atoms added/removed, substrate untouched).
- **Realistic:** the same substitution, *plus* explicit structural vacancies
  -- three separate mechanisms, each applied only to the site type it
  actually affects:

  - **O sites:** `TIO_VACANCY_FRAC ≈ 15%` -- rock-salt TiO's well-established
    intrinsic O-sublattice vacancy rate (it orders into a monoclinic Ti5O5
    superstructure below ~950 C; disordered/random vacancies above that --
    this notebook builds the disordered approximation). Applied at the
    **full** 15% to however many O sites currently exist, **not** scaled
    down by `x` -- it's a property of oxygen occupying that site, not of
    the bulk N/O ratio, so at `x=0.25` the 25% of anion sites that *are*
    oxygen should still see ~15% vacancy among themselves, not an
    "overall" ~3.75% diluted across the whole (mostly-still-nitrogen)
    anion sublattice.
  - **N sites:** `TIN_VACANCY_FRAC` (default 5%), present at *any* `x`
    including `x=0` (pure TiN). Unlike TiO's ~15%, there is no single
    established "TiN vacancy fraction" -- it's a genuinely wide,
    process-dependent range, not a fixed thermodynamic value at 1:1
    stoichiometry. Grounding the default in real numbers rather than a bare
    guess: real sputtered TiN films span N:Ti = 0.587-1.40 depending on
    deposition conditions, with film quality (lowest resistivity) peaking
    near N:Ti ≈ 1 (Chen *et al.*, https://doaj.org/article/30afc1597e7c4e5788321ea193fc85e5);
    separately, the N-deficient rock-salt phase is experimentally stable up
    to ~63% N-vacancy and predicted metastable to ~80% (Holec *et al.*,
    NANOCON 2020, https://www.confer.cz/nanocon/2020/read/3706-ab-initio-study-of-point-defects-formation-in-ti-tin-nanolayer.pdf).
    So `TIN_VACANCY_FRAC=0.05` is deliberately placed at the low,
    **well-optimized-film** end of a much wider possible range, not a
    "typical" or central value -- if you have a measured N:Ti ratio for
    your own material (RBS/XPS/EDS), use `TIN_VACANCY_FRAC = 1 - (N:Ti
    ratio)` directly instead of this placeholder.
  - **Ti sites:** `x * TIO_VACANCY_FRAC` -- treated as purely oxygen-driven
    (the literature attributes TiN's non-stoichiometry mainly to N
    vacancies, with the Ti sublattice close to fully occupied), so this
    still scales linearly from 0 at `x=0` to the full ~15% at `x=1`.

  These are real, explicit removed atoms (not fractional/partial-occupancy
  sites), since MD/MLIP training needs actual atomic positions.

**Correctness note (a real fix, not just a design choice):** an earlier
version of this function drew anion vacancies from the *combined* N+O pool
at a rate scaled by `x`, which sounds equivalent but isn't -- with the N and
O sites mixed together in one random draw, only about a fraction `x` of the
removed atoms would land on O by chance, so the O sites themselves ended up
with an *achieved* vacancy rate of only `x * 15%` (e.g. ~3.75% at `x=0.25`,
not the intended ~15%). Drawing O vacancies from the O sites exclusively (as
above) fixes that -- the O sublattice now hits the full ~15% at every `x`,
same as it does in pure TiO.

**A caveat, stated plainly, on both vacancy_frac constants:** `TIO_VACANCY_FRAC`
is a specific, well-established equilibrium value for TiO at ~1:1
stoichiometry. `TIN_VACANCY_FRAC` is not that kind of number -- it's a
single point (a well-optimized-film estimate, sourced above) chosen from a
real range that spans roughly 0-60%+ depending on process, so "realistic"
here means "grounded in real measurements of the *range*," not "the measured
value for TiN specifically" the way it is for TiO. Neither constant is a
literature-derived curve for the *mixed* oxynitride composition -- I don't
have a source for exactly how vacancy concentration evolves across the full
Ti(N,O) series. Treat the `realistic` outputs as "vacancy-aware," not
"quantitatively validated," and adjust `TIO_VACANCY_FRAC`/`TIN_VACANCY_FRAC` if your own XRD
density data says otherwise.

Also also built here: `build_tio_bulk_realistic()`, a standalone rock-salt
TiO reference cell with the same ~15%/15% vacancy treatment (useful as a
sanity check on the vacancy fraction actually achieved, independent of any
interface).

**Scope:** applied to both the narrow (`small_periodic_interfaces`, 4c) and
wide (`wide_periodic_interfaces`, 4d) MLIP-training cells, both terminations
-- via `build_oxynitride_variants()`, a thin loop wrapper reused for both so
the two sets are built identically. Built on the **periodic** (no-vacuum)
version, not the vacuum-slab `small_interfaces`/`wide_interfaces`, since
fully-periodic training data is what's actually wanted here; nothing about
`get_film_indices`/`substitute_film_anion` requires a vacuum gap to work, so
this substitutes/removes atoms directly on the already-periodic,
already-termination-matched cell -- atom positions never move, so
periodicity and the matched termination survive untouched by the N->O
substitution (species-only) and are only lightly perturbed by `realistic`'s
vacancy removal, which is realistic, not a bug. `build_oxynitride_interface()`
is still general, though; point it at `large_periodic_interfaces` the same
way if you want oxidized variants at the thermal-conductivity scale too.


In [ ]:
def remove_random_sites(structure, indices, frac, rng):
    """Randomly remove `frac` of the atoms in `indices` from `structure` --
    an explicit, physical defect (real atoms deleted), not a fractional/
    partial-occupancy site, since MD/MLIP training needs real atomic
    positions. Generic building block reused for both bulk vacancy
    concentration and film-restricted vacancy substitution below."""
    indices = list(indices)
    n_remove = round(len(indices) * frac)
    if n_remove == 0:
        return structure
    remove_idx = set(rng.choice(indices, size=n_remove, replace=False))
    keep = [i for i in range(len(structure)) if i not in remove_idx]
    all_species = structure.species  # cache -- see trim_top_layer note
    return Structure(structure.lattice.matrix.copy(),
                      [all_species[i] for i in keep],
                      structure.cart_coords[keep],
                      coords_are_cartesian=True)


def build_tio_bulk_realistic(a=TIO_A, vacancy_frac=TIO_VACANCY_FRAC,
                              repeats=(3, 3, 3), seed=RANDOM_SEED):
    """Rock-salt TiO (B1) with explicit structural vacancies on BOTH the Ti
    and O sublattices (~15% each in near-stoichiometric TiO). Built on a
    `repeats` supercell of the idealized 8-atom conventional cell so the
    requested vacancy fraction is statistically meaningful (the conventional
    cell alone only has 4 Ti + 4 O sites -- removing 15% of 4 isn't
    meaningful)."""
    ideal = Structure.from_spacegroup(
        "Fm-3m", Lattice.cubic(a), ["Ti", "O"], [[0, 0, 0], [0.5, 0, 0]]
    ) * repeats
    rng = np.random.default_rng(seed)
    ti_idx = [i for i, sp in enumerate(ideal.species) if sp.symbol == "Ti"]
    step1 = remove_random_sites(ideal, ti_idx, vacancy_frac, rng)
    o_idx = [i for i, sp in enumerate(step1.species) if sp.symbol == "O"]
    step2 = remove_random_sites(step1, o_idx, vacancy_frac, rng)
    return step2


TIO_BULK_REALISTIC_REPEATS = (3, 3, 3)  # supercell used by build_tio_bulk_realistic()
                                         # below, so the ~15% vacancy fraction is
                                         # statistically meaningful (the 8-atom
                                         # conventional cell alone is too small)

tio_bulk_idealized = Structure.from_spacegroup(
    "Fm-3m", Lattice.cubic(TIO_A), ["Ti", "O"], [[0, 0, 0], [0.5, 0, 0]]
)
tio_bulk_realistic = build_tio_bulk_realistic(repeats=TIO_BULK_REALISTIC_REPEATS)
n_ti_ideal = sum(1 for sp in tio_bulk_idealized.species if sp.symbol == "Ti")
n_ti_real = sum(1 for sp in tio_bulk_realistic.species if sp.symbol == "Ti")
n_o_real = sum(1 for sp in tio_bulk_realistic.species if sp.symbol == "O")
n_repeats = TIO_BULK_REALISTIC_REPEATS[0] * TIO_BULK_REALISTIC_REPEATS[1] * TIO_BULK_REALISTIC_REPEATS[2]
n_ti_full = n_ti_ideal * n_repeats
print(f"TiO idealized: {len(tio_bulk_idealized)} atoms, a={TIO_A} A")
print(f"TiO realistic: {len(tio_bulk_realistic)} atoms -- Ti vacancy frac "
      f"achieved = {1 - n_ti_real / n_ti_full:.3f} (target {TIO_VACANCY_FRAC}), "
      f"O vacancy frac achieved = {1 - n_o_real / n_ti_full:.3f} (target {TIO_VACANCY_FRAC})")


def get_film_indices(interface, tol=0.3):
    """Indices of all atoms belonging to the FILM (TiN) side of `interface`
    -- everything at or above the largest z-gap (the substrate/film
    interface), the same split classify_film_termination uses."""
    z = interface.cart_coords[:, 2]
    layers = get_z_layers(z, tol=tol)
    layer_z = [z[l].mean() for l in layers]
    gaps = np.diff(layer_z)
    split = int(np.argmax(gaps))
    return [i for l in layers[split + 1:] for i in l]


def substitute_film_anion(interface, x, rng, from_elem="N", to_elem="O", tol=0.3):
    """Randomly replace a fraction x of the FILM's `from_elem` sites with
    `to_elem` -- a species swap only (no position/lattice change); the
    substrate is never touched."""
    film_idx = get_film_indices(interface, tol=tol)
    all_symbols = [sp.symbol for sp in interface.species]  # cache -- see
                                                            # trim_top_layer note
    candidates = [i for i in film_idx if all_symbols[i] == from_elem]
    n_sub = round(len(candidates) * x)
    sub_idx = set(rng.choice(candidates, size=n_sub, replace=False)) if n_sub else set()
    species = [to_elem if i in sub_idx else all_symbols[i]
               for i in range(len(interface))]
    return Structure(interface.lattice.matrix.copy(), species,
                      interface.cart_coords.copy(), coords_are_cartesian=True)


def build_oxynitride_interface(interface, x, realistic,
                                o_vacancy_frac=TIO_VACANCY_FRAC,
                                n_vacancy_frac=TIN_VACANCY_FRAC,
                                seed=RANDOM_SEED, tol=0.3):
    """Build a TiN(1-x)O(x) film variant of `interface`. `realistic=False`:
    plain random N->O substitution on the film's anion sites. `realistic=
    True`: the same substitution, plus THREE separate vacancy mechanisms,
    each drawn only from the site type it actually affects (not a mixed
    pool -- see the 4e markdown for why that matters):
      - O sites: `o_vacancy_frac` (~15%, TiO's well-established intrinsic
        rate) -- NOT scaled by x, since it's a property of oxygen occupying
        the site, not of the bulk composition.
      - N sites: `n_vacancy_frac` (TiN's own baseline N-vacancy rate,
        default 5%, present at any x including x=0 -- pure TiN is not
        vacancy-free) -- also not scaled by x, same reasoning.
      - Ti sites: `x * o_vacancy_frac` -- treated as purely oxygen-driven
        (TiN's own Ti sublattice is close to fully occupied per the
        literature), so this term still scales linearly from 0 (x=0) to
        the full TiO Ti-vacancy rate (x=1).
    See the 4e markdown for the caveat that both vacancy_frac constants are
    approximations, not literature-derived curves for the mixed oxynitride
    composition specifically."""
    rng = np.random.default_rng(seed)
    subbed = substitute_film_anion(interface, x, rng, tol=tol)
    if not realistic:
        return subbed
    film_idx = get_film_indices(subbed, tol=tol)
    subbed_symbols = [sp.symbol for sp in subbed.species]  # cache -- see
                                                            # trim_top_layer note
    ti_idx = [i for i in film_idx if subbed_symbols[i] == "Ti"]
    with_ti_vac = remove_random_sites(subbed, ti_idx, x * o_vacancy_frac, rng)
    film_idx2 = get_film_indices(with_ti_vac, tol=tol)
    ti_vac_symbols = [sp.symbol for sp in with_ti_vac.species]
    o_idx = [i for i in film_idx2 if ti_vac_symbols[i] == "O"]
    with_o_vac = remove_random_sites(with_ti_vac, o_idx, o_vacancy_frac, rng)
    film_idx3 = get_film_indices(with_o_vac, tol=tol)
    o_vac_symbols = [sp.symbol for sp in with_o_vac.species]
    n_idx = [i for i in film_idx3 if o_vac_symbols[i] == "N"]
    return remove_random_sites(with_o_vac, n_idx, n_vacancy_frac, rng)


def build_oxynitride_variants(source_interfaces, x_points=OXIDATION_X_POINTS, label=""):
    """Build every (name, x, kind) TiN(1-x)O(x) variant of each interface in
    `source_interfaces` -- generic so it's reused for both the narrow (4c)
    and wide (4d) small periodic cells."""
    variants = {}
    for name, iface in source_interfaces.items():
        for x in x_points:
            for realistic, kind in [(False, "idealized"), (True, "realistic")]:
                variant = build_oxynitride_interface(iface, x, realistic)
                assert_no_overlaps(variant.cart_coords,
                                    label=f"{label}{name} x={x:.2f} {kind} oxynitride")
                variants[(name, x, kind)] = variant
                print(f"{label}{name} x={x:.2f} {kind}: {len(variant)} atoms, "
                      f"formula={variant.composition.reduced_formula}")
    return variants


oxynitride_variants = build_oxynitride_variants(small_periodic_interfaces)
wide_oxynitride_variants = build_oxynitride_variants(wide_periodic_interfaces, label="wide ")


## 4f. Realistic (vacancy-included) standard TiN/Si3N4 interfaces

Section 4e's `realistic` treatment already covers this at `x=0` -- pure TiN,
no oxygen, so only `TIN_VACANCY_FRAC` on the film's N sites applies (the O-
and Ti-vacancy terms are no-ops with no oxygen present). This section just
*applies* that `x=0` case to the plain (unoxidized) interfaces at each
scale, so "realistic" isn't something you only get by asking for partial
oxidation -- a **realistic, defect-included TiN/Si3N4 interface** (no
oxidation at all) is available as its own product, alongside the existing
defect-free ("idealized") version.

**Scope:** the three cells that are actually exported as training data --
`large_periodic_interfaces`, `small_periodic_interfaces`, and
`wide_periodic_interfaces` -- each get a `_realistic` counterpart. Applied
to the **already-tiled/large** cells, not the small unit cell before tiling
-- vacancies placed on the unit cell and then tiled would repeat
identically in every copy (an artificial periodic pattern, not how real
defects are distributed), so this draws independent random vacancy
placements across the full exported cell instead.


In [ ]:
def build_realistic_tin(source_interfaces, label=""):
    """Realistic (N-vacancy-included) counterpart of each interface in
    `source_interfaces` -- just the x=0 case of build_oxynitride_interface,
    reused as-is (no new logic): only TIN_VACANCY_FRAC applied to the
    film's N sites, since x=0 means no oxygen is present to trigger the O-
    or Ti-vacancy terms."""
    variants = {}
    for name, iface in source_interfaces.items():
        realistic = build_oxynitride_interface(iface, 0, realistic=True)
        assert_no_overlaps(realistic.cart_coords, label=f"{label}{name} realistic TiN")
        variants[name] = realistic
        print(f"{label}{name} realistic TiN: {len(realistic)} atoms "
              f"({len(iface) - len(realistic)} N vacancies removed)")
    return variants


large_periodic_interfaces_realistic = build_realistic_tin(large_periodic_interfaces, label="large ")
small_periodic_interfaces_realistic = build_realistic_tin(small_periodic_interfaces, label="small ")
wide_periodic_interfaces_realistic = build_realistic_tin(wide_periodic_interfaces, label="wide ")


## 5. Export (VASP POSCAR + extxyz)

`Structure.to(fmt="poscar")` doesn't group species in a specific order by
default. Rather than hand-roll that, convert to an `ase.Atoms` (via
`AseAtomsAdaptor`, needed for extxyz export anyway) and let
`ase.io.write(..., sort=True)` handle it -- ASE's own sort, well-tested,
one flag, no custom code.

Exports now go into a **folder tree that mirrors the variable names**, so a
path tells you what a file is without decoding a suffix chain like
`_mlip_wide_periodic_realistic_tin`:

```
interfaces_out/
  vacuum/{large,small,wide}/                       -- vacuum-slab results
  periodic/{large,small,wide}/idealized/           -- periodic base TiN, no vacancies
  periodic/{large,small,wide}/realistic/           -- periodic base TiN, N-vacancy included (4f)
  periodic/{small,wide}/oxynitride/                -- partial-oxidation TiN(1-x)O(x) variants (4e)
  periodic/tio_bulk/                               -- standalone rock-salt TiO references
```

`large` never gets an `oxynitride/` folder -- oxidation variants are only
built on the small/wide MLIP-training cells (see 4e), not the
thermal-conductivity-scale cells.

The pre-fix `_inequivalent` comparison structures (4b/4c/4d) are **no longer
exported to disk** -- now that the termination-matching fix is validated,
they were just debugging clutter next to real training data. They're still
built in memory (`large_periodic_interfaces_inequivalent` etc.) if you ever
want to diff one against its fixed counterpart yourself; just call
`export_structures(large_periodic_interfaces_inequivalent, "some/dir")`.

`Structure.to(fmt="poscar")` doesn't group species in a specific order by
default. Rather than hand-roll that, convert to an `ase.Atoms` (via
`AseAtomsAdaptor`, needed for extxyz export anyway) and let
`ase.io.write(..., sort=True)` handle it -- ASE's own sort, well-tested,
one flag, no custom code.


In [ ]:
import os

outdir = "interfaces_out"


def d(*parts):
    """Make (if needed) and return outdir/parts... -- one call per leaf
    folder in the tree described above."""
    path = os.path.join(outdir, *parts)
    os.makedirs(path, exist_ok=True)
    return path


vacuum_large_dir              = d("vacuum", "large")
vacuum_small_dir               = d("vacuum", "small")
vacuum_wide_dir                 = d("vacuum", "wide")
periodic_large_idealized_dir  = d("periodic", "large", "idealized")
periodic_large_realistic_dir  = d("periodic", "large", "realistic")
periodic_small_idealized_dir  = d("periodic", "small", "idealized")
periodic_small_realistic_dir  = d("periodic", "small", "realistic")
periodic_small_oxynitride_dir = d("periodic", "small", "oxynitride")
periodic_wide_idealized_dir   = d("periodic", "wide", "idealized")
periodic_wide_realistic_dir   = d("periodic", "wide", "realistic")
periodic_wide_oxynitride_dir  = d("periodic", "wide", "oxynitride")
periodic_tio_bulk_dir         = d("periodic", "tio_bulk")

adaptor = AseAtomsAdaptor()
summary = []


def export_structures(structures, target_dir, tag_fn=lambda key: key.replace("-", "_")):
    """Write every {key: Structure} in `structures` to POSCAR + extxyz in
    `target_dir`. `tag_fn(key)` builds the filename tag from the dict key --
    override it for non-plain-name keys (e.g. the oxynitride variants'
    (name, x, kind) tuples, via oxynitride_tag below)."""
    for key, iface in structures.items():
        tag = tag_fn(key)
        poscar_path = os.path.join(target_dir, f"POSCAR_{tag}")
        xyz_path = os.path.join(target_dir, f"interface_{tag}.extxyz")

        atoms = adaptor.get_atoms(iface)
        write(poscar_path, atoms, format="vasp", direct=True, sort=True)
        write(xyz_path, atoms, format="extxyz")

        counts = {el: iface.composition.get(el, 0) for el in ("Ti", "Si", "N", "O")}
        summary.append(dict(path=os.path.relpath(poscar_path, outdir), natoms=len(iface),
                             cell=tuple(round(c, 3) for c in iface.lattice.parameters[:3]),
                             **counts))
        print(f"wrote {poscar_path}  and  {xyz_path}")


def oxynitride_tag(key):
    name, x, kind = key
    return f"{name.replace('-', '_')}_x{x:.2f}_{kind}"


# --- vacuum-slab results ---
export_structures(large_interfaces, vacuum_large_dir)
export_structures(small_interfaces, vacuum_small_dir)
export_structures(wide_interfaces, vacuum_wide_dir)

# --- fully periodic (no-vacuum) base TiN: idealized vs. realistic (4f) ---
export_structures(large_periodic_interfaces, periodic_large_idealized_dir)
export_structures(large_periodic_interfaces_realistic, periodic_large_realistic_dir)
export_structures(small_periodic_interfaces, periodic_small_idealized_dir)
export_structures(small_periodic_interfaces_realistic, periodic_small_realistic_dir)
export_structures(wide_periodic_interfaces, periodic_wide_idealized_dir)
export_structures(wide_periodic_interfaces_realistic, periodic_wide_realistic_dir)

# --- partial-oxidation variants (4e) -- only built on the small/wide MLIP
# cells, not the large thermal-conductivity-scale ones ---
export_structures(oxynitride_variants, periodic_small_oxynitride_dir, tag_fn=oxynitride_tag)
export_structures(wide_oxynitride_variants, periodic_wide_oxynitride_dir, tag_fn=oxynitride_tag)

# --- standalone rock-salt TiO bulk references (always periodic) ---
export_structures(
    {"TiO_bulk_idealized": tio_bulk_idealized, "TiO_bulk_realistic": tio_bulk_realistic},
    periodic_tio_bulk_dir,
)

# NOTE: the pre-fix `_inequivalent` comparison dicts from 4b/4c/4d
# (large/small/wide_periodic_interfaces_inequivalent) are deliberately NOT
# exported here -- see the markdown above. They're still available in memory
# if you want to export or inspect one yourself, e.g.:
#   export_structures(large_periodic_interfaces_inequivalent, "scratch_dir")

print()
for row in summary:
    print(row)


## 6. Quick visualization

Plots **everything currently built**, but consolidated into 7 grids, not one
per result dict -- each scale (unit-cell, large, small, wide) merges its
vacuum/periodic/realistic variants into a single grid via `merge_variants()`,
rather than a separate image per variant, so this doesn't balloon by one
whole picture every time a new variant gets added. The `_inequivalent`
pre-fix comparisons from 4b/4c/4d are **not** auto-plotted here (they were a
one-time debugging aid, not something you need to eyeball on every run) --
call `plot_structure_grid(large_periodic_interfaces_inequivalent, ...)`
yourself if you want to look at one again; the dicts are still built and
exported, nothing about the data changed. `plot_structure_grid()` and
`merge_variants()` are both generic -- extend `plot_categories` the same way
if you build more result dicts later.

**Speed:** the actual bottleneck is `ase.visualize.plot.plot_atoms` itself --
it draws one matplotlib circle patch per atom, and that scales badly once a
structure gets into the thousands (the `large_*` cells are tiled up to
`TARGET_ATOMS=5000`+). Rather than dropping categories to cut runtime,
`plot_structure_grid()` crops any structure over `max_atoms` down to a
spatially contiguous patch (shrinks the in-plane box around the origin, not
an arbitrary index slice) before handing it to `plot_atoms` -- so every panel
renders fast regardless of the underlying cell's real size, and all
categories stay in the preview. The crop is clearly labeled in each panel's
title (`N/total atoms, cropped`) so it's never mistaken for the actual
exported structure -- POSCAR/extxyz exports in section 5 are never cropped,
only these preview plots.


In [ ]:
from ase.visualize.plot import plot_atoms
import matplotlib.pyplot as plt


def crop_for_preview(structure, max_atoms):
    """Return a spatially contiguous sub-region of `structure` with about
    `max_atoms` atoms (shrinks the in-plane a/b box around the origin, so
    the crop is a real patch of the periodic structure, not an arbitrary
    index slice that could look broken). Only used for the preview plots
    below -- never touches the actual exported structures."""
    if len(structure) <= max_atoms:
        return structure, len(structure)
    frac_ab = structure.frac_coords[:, :2]
    keep_frac = min(1.0, (max_atoms / len(structure)) ** 0.5)
    idx = np.where((frac_ab[:, 0] < keep_frac) & (frac_ab[:, 1] < keep_frac))[0]
    if len(idx) == 0:
        idx = np.arange(min(max_atoms, len(structure)))
    all_species = structure.species  # cache -- see trim_top_layer note
    cropped = Structure(structure.lattice.matrix.copy(),
                         [all_species[i] for i in idx],
                         structure.cart_coords[idx],
                         coords_are_cartesian=True)
    return cropped, len(structure)


def plot_structure_grid(structures, title, filename, ncols=4, figsize_per=(3.2, 3.6),
                         max_atoms=250):
    """Plot every (name, Structure) in `structures` as a small ASE-rendered
    panel in a grid, save to `filename` in `outdir`, and show inline.
    Structures over `max_atoms` are cropped for speed -- see crop_for_preview."""
    items = list(structures.items())
    if not items:
        print(f"  (nothing to plot for {title})")
        return
    ncols = min(ncols, len(items))
    nrows = -(-len(items) // ncols)  # ceil
    fig, axes = plt.subplots(nrows, ncols,
                              figsize=(figsize_per[0] * ncols, figsize_per[1] * nrows),
                              squeeze=False)
    for ax, (name, struct) in zip(axes.flat, items):
        cropped, n_full = crop_for_preview(struct, max_atoms)
        plot_atoms(adaptor.get_atoms(cropped), ax, radii=0.4, rotation="-90x,0y,0z")
        label = (f"{name}\n({n_full} atoms)" if len(cropped) == n_full else
                 f"{name}\n({len(cropped)}/{n_full} atoms, cropped)")
        ax.set_title(label, fontsize=8)
    for ax in axes.flat[len(items):]:
        ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    path = os.path.join(outdir, filename)
    plt.savefig(path, dpi=150)
    plt.show()
    print(f"  saved {path}")


def merge_variants(**named_dicts):
    """Merge several {name: Structure} dicts into one grid-able dict, tagging
    each item's key with the dict's own label (e.g. merge_variants(vacuum=d1,
    periodic=d2) -> {"Ti-term (vacuum)": ..., "Ti-term (periodic)": ...}).
    Lets one grid show a scale's vacuum/periodic/realistic variants side by
    side instead of needing a separate picture per variant."""
    merged = {}
    for label, structures in named_dicts.items():
        for name, struct in structures.items():
            merged[f"{name} ({label})"] = struct
    return merged


oxynitride_labeled = {f"{name} x={x:.2f} {kind}": struct
                       for (name, x, kind), struct in oxynitride_variants.items()}
wide_oxynitride_labeled = {f"{name} x={x:.2f} {kind}": struct
                            for (name, x, kind), struct in wide_oxynitride_variants.items()}

plot_categories = [
    (merge_variants(vacuum=interfaces, periodic=periodic_interfaces),
     "Unit-cell interfaces", "preview_unit_cell.png"),
    (merge_variants(vacuum=large_interfaces, periodic=large_periodic_interfaces,
                     periodic_realistic=large_periodic_interfaces_realistic),
     "Large cells -- thermal-conductivity scale", "preview_large.png"),
    (merge_variants(vacuum=small_interfaces, periodic=small_periodic_interfaces,
                     periodic_realistic=small_periodic_interfaces_realistic),
     "Small MLIP-training cells", "preview_small.png"),
    (merge_variants(vacuum=wide_interfaces, periodic=wide_periodic_interfaces,
                     periodic_realistic=wide_periodic_interfaces_realistic),
     "Wide small MLIP-training cells", "preview_wide.png"),
    (oxynitride_labeled, "Partial-oxidation TiN(1-x)O(x) variants", "preview_oxynitride.png"),
    (wide_oxynitride_labeled, "Wide partial-oxidation TiN(1-x)O(x) variants", "preview_oxynitride_wide.png"),
    ({"TiO bulk idealized": tio_bulk_idealized, "TiO bulk realistic (vacancies)": tio_bulk_realistic},
     "Rock-salt TiO bulk references", "preview_tio_bulk.png"),
]

for structures, title, filename in plot_categories:
    print(title)
    plot_structure_grid(structures, title, filename)
